# his-bench downstream results

Reads results directly from the SLURM job logs written by `scripts/downstream/run_pangaea_consistency_cdl.sh` (UPerNet) and `scripts/downstream/run_pangaea_consistency_mlp_fusion.sh` (MLPFusionHead), and builds one task x encoder table per decoder.

**Why logs, not `sacct`/checkpoint existence:** a SLURM job reporting `COMPLETED` does not mean training actually finished (we hit this directly -- a device-count mismatch crashed a run in under a minute while SLURM still reported success), and a saved checkpoint file exists from the very first validation epoch onward (`ModelCheckpoint` overwrites the best-so-far checkpoint continuously), not just at the end. The only reliable signal is the `Best ckpt path:` line `train.py` prints, which only appears after `trainer.test()` runs -- i.e. after all epochs of `trainer.fit()` finish.

**"Best result":** for a given (task, backbone, decoder), this notebook globs *every* completed run directory found (there can be more than one -- e.g. a rerun after a timeout, or an accidental duplicate submission) and keeps the one with the highest test mIoU. Re-run the cells below any time to pick up new results as jobs finish.

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd

## Configuration

Mirrors the exact naming scheme used by both `run_pangaea_consistency_*.sh` scripts (task/sensor map, weight_init tags, protocol overrides, experiment_setup suffix per decoder). If either script's naming changes, update it here too.

In [2]:
DISK_ROOT = Path("/dss/dsstbyfs02/pn49cu/pn49cu-dss-0001/niklas/experiments/hsi-bench")

TASKS = ["cdl", "bdforet", "bnetd", "eurocrops", "nlcd", "treemap", "desis_cdl", "eo1_cdl"]
TASK_LABELS = {
    "cdl": "CDL",
    "bdforet": "BD-F",
    "bnetd": "BNTD",
    "eurocrops": "EuCrops",
    "nlcd": "NLCD",
    "treemap": "TreeMap",
    "desis_cdl": "D-CDL",
    "eo1_cdl": "EO1-CDL",
}

BACKBONES = ["dofa_b", "dofav2_b", "copernicus_fm_b", "panopticon_b", "carl", "vit_b", "vit_b_finetune"]
BACKBONE_LABELS = {
    "dofa_b": "DOFA-B",
    "dofav2_b": "DOFAv2-B",
    "copernicus_fm_b": "Copernicus-FM-B",
    "panopticon_b": "Panopticon-B",
    "carl": "CARL",
    "vit_b": "ViT-B (random, frozen)",
    "vit_b_finetune": "ViT-B (random, supervised)",
}

# Source sensor per task -- must match task_sensor_map in run_pangaea_consistency_cdl.sh
TASK_SENSOR = {
    "cdl": "enmap", "bdforet": "enmap", "bnetd": "enmap", "eurocrops": "enmap",
    "nlcd": "enmap", "treemap": "enmap", "desis_cdl": "desis", "eo1_cdl": "eo1",
}

# weight_init tag per backbone -- must match weight_init_tag_map in both scripts
WEIGHT_INIT = {
    "dofa_b": "dofa_pretrained",
    "dofav2_b": "dofav2_pretrained",
    "copernicus_fm_b": "copernicus_fm_pretrained",
    "panopticon_b": "panopticon_pretrained",
    "carl": "carl_pretrained",
    "vit_b": "random_init",
    "vit_b_finetune": "random_init_finetune",
}

# protocol label per backbone -- default "linear"; only vit_b_finetune differs
PROTOCOL = {"vit_b_finetune": "finetune"}
DEFAULT_PROTOCOL = "linear"

# decoder folder name -> experiment_setup suffix each script uses
# ("<task>_<suffix>"), e.g. "cdl_upernet_linear" / "cdl_mlpfusion_linear"
DECODER_SETUP_SUFFIX = {
    "upernet_head": "upernet_linear",
    "mlp_fusion_head": "mlpfusion_linear",
}
DECODER_LABELS = {
    "upernet_head": "UPerNet",
    "mlp_fusion_head": "MLPFusion (bilinear)",
}

## Result extraction

In [3]:
_METRIC_PATTERNS = {
    "mIoU": re.compile(r"test_MulticlassJaccardIndex\s*\S*\s*([0-9]+\.[0-9]+)"),
    "accuracy": re.compile(r"test_MulticlassAccuracy\s*\S*\s*([0-9]+\.[0-9]+)"),
    "loss": re.compile(r"test_loss\s*\S*\s*([0-9]+\.[0-9]+)"),
}
_CKPT_PATTERN = re.compile(r"Best ckpt path:\s*(\S+)")
_EPOCH_PATTERN = re.compile(r"epoch_(\d+)\.ckpt")


def parse_run_log(log_path: Path) -> dict | None:
    """Parse one .submitit/*/*_0_log.out file. Returns None if the run never
    reached trainer.test() (i.e. training didn't actually finish), regardless
    of what SLURM's own exit status says."""
    text = log_path.read_text(errors="ignore")
    if "Best ckpt path:" not in text:
        return None

    result: dict = {"log_path": str(log_path)}
    for name, pattern in _METRIC_PATTERNS.items():
        matches = pattern.findall(text)
        if not matches:
            return None  # completion line present but metrics missing -- treat as unusable
        result[name] = float(matches[-1])

    ckpt_match = _CKPT_PATTERN.search(text)
    if ckpt_match:
        result["checkpoint"] = ckpt_match.group(1)
        epoch_match = _EPOCH_PATTERN.search(ckpt_match.group(1))
        result["best_epoch"] = int(epoch_match.group(1)) if epoch_match else None
    return result


def find_best_result(task: str, backbone: str, decoder: str) -> dict | None:
    """Find every completed run for (task, backbone, decoder) and return the
    one with the highest test mIoU. There can legitimately be more than one
    multirun directory here (a rerun after a timeout, or an accidental
    duplicate submission) -- picking the best rather than e.g. the most
    recent avoids silently reporting a stale/failed rerun."""
    experiment_setup = f"{task}_{DECODER_SETUP_SUFFIX[decoder]}"
    sensor = TASK_SENSOR[task]
    weight_init = WEIGHT_INIT[backbone]
    protocol = PROTOCOL.get(backbone, DEFAULT_PROTOCOL)

    task_dir = (
        DISK_ROOT / "downstream" / experiment_setup / sensor / decoder
        / backbone / protocol / weight_init
    )
    log_files = sorted(task_dir.glob("multiruns/*/.submitit/*/*_0_log.out"))

    candidates = [parsed for f in log_files if (parsed := parse_run_log(f)) is not None]
    if not candidates:
        return None
    return max(candidates, key=lambda c: c["mIoU"])

## Build the long-format results table

One row per (decoder, task, backbone) with every metric -- the source of truth the pivoted tables below are built from. Also useful on its own for spotting which combinations are still missing/running.

In [4]:
rows = []
for decoder in DECODER_SETUP_SUFFIX:
    for task in TASKS:
        for backbone in BACKBONES:
            result = find_best_result(task, backbone, decoder)
            rows.append({
                "decoder": DECODER_LABELS[decoder],
                "task": TASK_LABELS[task],
                "backbone": BACKBONE_LABELS[backbone],
                "mIoU": result["mIoU"] if result else None,
                "accuracy": result["accuracy"] if result else None,
                "loss": result["loss"] if result else None,
                "best_epoch": result["best_epoch"] if result else None,
                "status": "done" if result else "missing/running",
            })

results_long = pd.DataFrame(rows)

missing = results_long[results_long["status"] == "missing/running"]
print(f"{len(results_long) - len(missing)} / {len(results_long)} combinations completed.")
if len(missing):
    print("Missing or still running:")
    display(missing[["decoder", "task", "backbone"]])

110 / 112 combinations completed.
Missing or still running:


,decoder,task,backbone
87,MLPFusion (bilinear),NLCD,Panopticon-B
88,MLPFusion (bilinear),NLCD,CARL


## Pivoted tables: task (columns) x encoder (rows)

Test mIoU, best result per cell. Bold + shaded = best encoder for that task.

In [5]:
def pivot_table(decoder: str, metric: str = "mIoU") -> pd.DataFrame:
    subset = results_long[results_long["decoder"] == DECODER_LABELS[decoder]]
    table = subset.pivot(index="backbone", columns="task", values=metric)
    row_order = [BACKBONE_LABELS[b] for b in BACKBONES]
    col_order = [TASK_LABELS[t] for t in TASKS]
    return table.reindex(index=row_order, columns=col_order)


def style_table(df: pd.DataFrame, caption: str):
    return (
        df.style
        .format("{:.3f}", na_rep="\u2014")
        .highlight_max(axis=0, props="font-weight:bold;background-color:#e6f4ea;")
        .set_caption(caption)
    )

In [6]:
upernet_table = pivot_table("upernet_head")
style_table(upernet_table, "UPerNet -- test mIoU")

task,CDL,BD-F,BNTD,EuCrops,NLCD,TreeMap,D-CDL,EO1-CDL
backbone,,,,,,,,
DOFA-B,0.530,0.459,0.347,0.419,0.301,0.324,0.534,0.572
DOFAv2-B,0.501,0.451,0.345,0.420,0.295,0.316,0.515,0.573
Copernicus-FM-B,0.528,0.451,0.347,0.422,0.295,0.321,0.526,0.573
Panopticon-B,0.587,0.583,0.378,0.478,0.382,0.387,0.560,0.624
CARL,0.593,0.570,0.390,0.501,0.350,0.368,0.613,0.572
"ViT-B (random, frozen)",0.548,0.521,0.352,0.449,0.316,0.364,0.529,0.551
"ViT-B (random, supervised)",0.683,0.661,0.394,0.548,0.411,0.410,0.613,0.636


In [7]:
mlp_fusion_table = pivot_table("mlp_fusion_head")
style_table(mlp_fusion_table, "MLPFusion (bilinear) -- test mIoU")

task,CDL,BD-F,BNTD,EuCrops,NLCD,TreeMap,D-CDL,EO1-CDL
backbone,,,,,,,,
DOFA-B,0.580,0.495,0.356,0.452,0.348,0.355,0.573,0.570
DOFAv2-B,0.515,0.468,0.354,0.429,0.347,0.345,0.549,0.557
Copernicus-FM-B,0.571,0.468,0.356,0.449,0.348,0.353,0.563,0.569
Panopticon-B,0.622,0.621,0.387,0.507,—,0.396,0.586,0.631
CARL,0.599,0.544,0.387,0.488,—,0.374,0.614,0.521
"ViT-B (random, frozen)",0.391,0.408,0.298,0.349,0.314,0.318,0.363,0.376
"ViT-B (random, supervised)",0.573,0.571,0.367,0.468,0.398,0.394,0.540,0.548


## Ranking: how each encoder stacks up per task

Per-task rank of each backbone (1 = best mIoU on that task), plus an **Avg Rank** column (mean rank across all tasks) to summarize overall standing across the whole benchmark. Sorted by Avg Rank, so the strongest encoder overall is at the top. A missing/still-running combination is left unranked (`—`), not silently dropped or treated as worst.

In [8]:
def rank_table(decoder: str, metric: str = "mIoU", higher_is_better: bool = True) -> pd.DataFrame:
    """Per-task rank of each backbone (1 = best), plus an Avg Rank column
    (mean rank across tasks) summarizing overall standing. Ties share the
    same (minimum) rank. Missing/still-running combinations stay NaN --
    excluded from Avg Rank via pandas' default skipna, not counted as a
    last-place finish."""
    values = pivot_table(decoder, metric=metric)
    ranks = values.rank(axis=0, ascending=not higher_is_better, method="min")
    ranks["Avg Rank"] = ranks.mean(axis=1).round(2)
    return ranks.sort_values("Avg Rank")


def style_rank_table(df: pd.DataFrame, caption: str):
    task_cols = [c for c in df.columns if c != "Avg Rank"]
    n_backbones = len(BACKBONES)
    return (
        df.style
        .format("{:.0f}", subset=task_cols, na_rep="—")
        .format("{:.2f}", subset=["Avg Rank"], na_rep="—")
        .background_gradient(cmap="RdYlGn_r", subset=task_cols, vmin=1, vmax=n_backbones)
        .background_gradient(cmap="RdYlGn_r", subset=["Avg Rank"], vmin=1, vmax=n_backbones)
        .set_caption(caption)
    )

In [9]:
upernet_ranks = rank_table("upernet_head")
style_rank_table(upernet_ranks, "UPerNet -- rank per task (1 = best), sorted by Avg Rank")

task,CDL,BD-F,BNTD,EuCrops,NLCD,TreeMap,D-CDL,EO1-CDL,Avg Rank
backbone,,,,,,,,,
"ViT-B (random, supervised)",1,1,1,1,1,1,1,1,1.00
Panopticon-B,3,2,3,3,2,2,3,2,2.50
CARL,2,3,2,2,3,3,2,6,2.88
"ViT-B (random, frozen)",4,4,4,4,4,4,5,7,4.50
DOFA-B,5,5,6,7,5,5,4,5,5.25
Copernicus-FM-B,6,7,5,5,7,6,6,3,5.62
DOFAv2-B,7,6,7,6,6,7,7,4,6.25


In [10]:
mlp_fusion_ranks = rank_table("mlp_fusion_head")
style_rank_table(mlp_fusion_ranks, "MLPFusion (bilinear) -- rank per task (1 = best), sorted by Avg Rank")

task,CDL,BD-F,BNTD,EuCrops,NLCD,TreeMap,D-CDL,EO1-CDL,Avg Rank
backbone,,,,,,,,,
Panopticon-B,1,1,2,1,—,1,2,1,1.29
CARL,2,3,1,2,—,3,1,6,2.57
"ViT-B (random, supervised)",4,2,3,3,1,2,6,5,3.25
DOFA-B,3,4,5,4,2,4,3,2,3.38
Copernicus-FM-B,5,6,4,5,3,5,4,3,4.38
DOFAv2-B,6,5,6,6,4,6,5,4,5.25
"ViT-B (random, frozen)",7,7,7,7,5,7,7,7,6.75


## Optional: same tables for accuracy / loss

Swap `metric=` below for anything in `results_long.columns` (`accuracy`, `loss`, `best_epoch`).

In [11]:
style_table(pivot_table("upernet_head", metric="accuracy"), "UPerNet -- test accuracy")

task,CDL,BD-F,BNTD,EuCrops,NLCD,TreeMap,D-CDL,EO1-CDL
backbone,,,,,,,,
DOFA-B,0.692,0.629,0.515,0.591,0.463,0.489,0.696,0.728
DOFAv2-B,0.668,0.622,0.513,0.592,0.455,0.480,0.680,0.729
Copernicus-FM-B,0.691,0.622,0.516,0.593,0.455,0.487,0.689,0.729
Panopticon-B,0.740,0.737,0.549,0.647,0.553,0.558,0.718,0.769
CARL,0.745,0.726,0.561,0.668,0.518,0.538,0.760,0.727
"ViT-B (random, frozen)",0.708,0.685,0.521,0.620,0.480,0.534,0.692,0.710
"ViT-B (random, supervised)",0.812,0.796,0.565,0.708,0.583,0.581,0.760,0.777


In [12]:
style_table(pivot_table("mlp_fusion_head", metric="accuracy"), "MLPFusion (bilinear) -- test accuracy")

task,CDL,BD-F,BNTD,EuCrops,NLCD,TreeMap,D-CDL,EO1-CDL
backbone,,,,,,,,
DOFA-B,0.734,0.662,0.525,0.623,0.517,0.524,0.729,0.726
DOFAv2-B,0.680,0.637,0.523,0.601,0.515,0.513,0.709,0.715
Copernicus-FM-B,0.727,0.637,0.525,0.620,0.516,0.522,0.720,0.725
Panopticon-B,0.767,0.766,0.558,0.672,—,0.568,0.739,0.774
CARL,0.749,0.704,0.558,0.656,—,0.545,0.761,0.685
"ViT-B (random, frozen)",0.562,0.579,0.460,0.517,0.478,0.482,0.532,0.547
"ViT-B (random, supervised)",0.729,0.727,0.537,0.637,0.570,0.565,0.701,0.708
